# Code mẫu — Ensemble

Muốn gộp nhiều model, các dự đoán phải nói về cùng một mẫu và cùng một lớp. Cách gộp cũng phải hợp với metric của đề. Bốn ví dụ dưới đây đi từ phép lấy trung bình tính tay, tới gộp xác suất đã hiệu chỉnh, gộp theo thứ hạng, và cuối cùng là stacking với đặc trưng out-of-fold.

## Ví dụ 1: Trung bình xác suất có thể khác bỏ phiếu đa số

Ba model cùng dự đoán xác suất lớp 1 cho bốn mẫu. Ở mẫu đầu, hai model cho 0.51, còn model thứ ba cho 0.01. Ta so hai cách gộp: bỏ phiếu đa số theo nhãn, và lấy trung bình xác suất. Trung bình được kiểm lại bằng một vòng tính từng hàng.

In [ ]:
import numpy as np

P = np.array([[0.51, 0.51, 0.01], [0.9, 0.6, 0.6],
              [0.2, 0.8, 0.7], [0.8, 0.1, 0.1]])
assert P.shape == (4, 3)  # Hàng là mẫu, cột là model. Mọi cột đều là xác suất lớp 1.
hard = ((P >= 0.5).sum(axis=1) >= 2).astype(int)
mean = P.mean(axis=1)
manual = np.array([sum(row) / len(row) for row in P])
assert np.allclose(mean, manual)
print("majority:", hard.tolist())
print("mean xác suất:", mean.round(3).tolist())
print("nhãn từ mean:", (mean >= 0.5).astype(int).tolist())

# Cùng một mẫu, nhưng model B lưu các cột lớp theo thứ tự ngược lại.
A = np.array([0.8, 0.2])
B, classes_B = np.array([0.3, 0.7]), [1, 0]
aligned_B = B[[classes_B.index(label) for label in [0, 1]]]
print("mean chưa/sau căn lớp:", ((A + B) / 2).round(2).tolist(),
      ((A + aligned_B) / 2).round(2).tolist())
assert np.allclose((A + aligned_B) / 2, [0.75, 0.25])

majority: [1, 1, 1, 0]
mean xác suất: [0.343, 0.7, 0.567, 0.333]
nhãn từ mean: [0, 1, 1, 0]
mean chưa/sau căn lớp: [0.55, 0.45] [0.75, 0.25]


Ở mẫu đầu, hai model nghiêng nhẹ về lớp 1, nên bỏ phiếu đa số chọn 1. Nhưng model thứ ba rất chắc chắn là lớp 0, nên trung bình xác suất chỉ còn 0.343, và kết quả là 0. Bỏ phiếu chỉ đếm hướng nghiêng. Trung bình xác suất tính cả độ chắc chắn.

Phần sau là một lỗi dễ mắc. Model B lưu cột lớp 1 trước, cột lớp 0 sau. Cộng thẳng, ta được `[0.55, 0.45]`. Căn lại thứ tự lớp rồi mới cộng, ta được `[0.75, 0.25]`. Cả hai kết quả đều có tổng bằng 1, nên kiểm tổng xác suất không phát hiện được lỗi này. Phải căn đúng ID và đúng lớp trước, rồi mới lấy trung bình.

## Ví dụ 2: Gộp xác suất sau khi hiệu chỉnh trên dữ liệu train

Dữ liệu là hai hình bán nguyệt lồng nhau, 240 mẫu, chia thành 144 train, 48 validation, và 48 mẫu kiểm cuối, gọi là audit. Hai model là logistic regression và cây quyết định độ sâu 3. Mỗi model được hiệu chỉnh xác suất bằng hàm sigmoid, qua ba fold nằm **bên trong train**. Trọng số gộp được cố định là bằng nhau ngay từ đầu. Validation và audit không tham gia học model hay học bước hiệu chỉnh.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss

X, y = make_moons(n_samples=240, noise=0.25, random_state=84)
train_rows, held_rows = train_test_split(np.arange(len(y)), test_size=0.4,
                                        random_state=1, stratify=y)
val_rows, audit_rows = train_test_split(held_rows, test_size=0.5,
                                       random_state=2, stratify=y[held_rows])
assert not (set(train_rows) & set(held_rows))
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=3)
estimators = {
    "linear": make_pipeline(StandardScaler(), LogisticRegression(max_iter=300)),
    "tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=4, random_state=4)
}
models = {}
for name, estimator in estimators.items():
    calibrated = CalibratedClassifierCV(estimator, method="sigmoid", cv=cv,
                                         ensemble=False, n_jobs=1)
    calibrated.fit(X[train_rows], y[train_rows])
    assert calibrated.classes_.tolist() == [0, 1]
    models[name] = calibrated
print("train/validation/audit:", len(train_rows), len(val_rows), len(audit_rows))
for split_name, rows in [("validation", val_rows), ("audit", audit_rows)]:
    predictions = np.stack([model.predict_proba(X[rows]) for model in models.values()])
    assert predictions.shape == (2, 48, 2)  # Trục: model, mẫu, lớp.
    average = predictions.mean(axis=0)
    assert np.allclose(average.sum(axis=1), 1)
    for name, probability in list(zip(models, predictions)) + [("mean", average)]:
        print(split_name, name, "log-loss:", f"{log_loss(y[rows], probability):.4f}")

train/validation/audit: 144 48 48
validation linear log-loss: 0.4037
validation tree log-loss: 0.3654
validation mean log-loss: 0.3666
audit linear log-loss: 0.3194
audit tree log-loss: 0.3748
audit mean log-loss: 0.3235


Mảng dự đoán có shape `(2, 48, 2)`, với ba trục rõ ràng: model, mẫu, lớp. Lấy trung bình theo trục 0, tức trục model, ta còn một phân phối xác suất cho mỗi mẫu. Với `ensemble=False`, bước hiệu chỉnh học từ dự đoán chéo giữa các fold trong train, rồi model gốc được học lại trên toàn bộ train.

Trên validation, cây tốt nhất với 0.3654, bản gộp sát sau với 0.3666. Trên audit thì logistic tốt nhất với 0.3194, cây lại kém nhất. Bản gộp không thắng model tốt nhất ở tập nào, nhưng không bao giờ là model tệ nhất. Trên những tập nhỏ như thế này, thứ hạng giữa các model có thể đảo lộn, và gộp là cách giảm rủi ro chọn nhầm. Audit chỉ được đọc sau khi đã chốt cách gộp, không dùng để chỉnh lại trọng số.

## Ví dụ 3: Gộp theo thứ hạng khi đề chấm thứ tự

Tám mẫu có nhãn thật là `0, 0, 1, 0, 1, 0, 1, 1`. Model A cho điểm trên thang khoảng 0 tới 1, model B trên thang 0 tới 10, và hai model xếp sai ở những mẫu khác nhau. AUC chỉ hỏi một điều: mẫu dương có được xếp trên mẫu âm hay không. Đổi điểm của mỗi model thành thứ hạng rồi lấy trung bình, ta bỏ được ảnh hưởng của thang điểm. Nhưng con số thu được không còn là xác suất.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

y = np.array([0, 0, 1, 0, 1, 0, 1, 1])
scores = np.array([[0.1, 3.0], [0.2, 2.0], [0.7, 8.0], [0.4, 1.0],
                   [0.8, 9.0], [0.3, 4.0], [0.35, 6.0], [0.9, 2.5]])
ranks = pd.DataFrame(scores).rank(axis=0, method="average").to_numpy()
assert np.array_equal(ranks[:, 1], pd.Series(scores[:, 1] * 10).rank().to_numpy())
combined = ((ranks - 0.5) / len(y)).mean(axis=1)
positive, negative = combined[y == 1], combined[y == 0]
manual_auc = np.mean([float(p > n) + 0.5 * float(p == n)
                      for p in positive for n in negative])
assert np.isclose(manual_auc, roc_auc_score(y, combined))
print("rank gộp:", combined.round(3).tolist())
print("AUC A/B/rank:", [round(roc_auc_score(y, s), 3)
                        for s in [scores[:, 0], scores[:, 1], combined]])
print("số cặp dương-âm kiểm:", len(positive) * len(negative))

rank gộp: [0.25, 0.188, 0.75, 0.312, 0.875, 0.438, 0.562, 0.625]
AUC A/B/rank: [0.938, 0.875, 1.0]
số cặp dương-âm kiểm: 16


AUC được kiểm lại bằng tay: xét đủ 16 cặp một mẫu dương với một mẫu âm, cặp nào xếp đúng được 1 điểm, cặp hòa được nửa điểm. Model A xếp sai một cặp, model B sai hai cặp, nhưng là những cặp khác nhau. Gộp thứ hạng sửa được cả ba, và đạt AUC 1. Dòng assert thứ hai kiểm rằng nhân điểm của B lên 10 lần cũng không đổi thứ hạng, nên không model nào lấn át chỉ vì thang điểm lớn hơn.

AUC bằng 1 trên tám mẫu tự đặt chưa nói gì về dữ liệu mới. Thứ hạng giữ thứ tự nhưng bỏ mất khoảng cách giữa các điểm. Đừng gọi một thứ hạng cao là “xác suất 90%”, và đừng nghĩ log-loss cũng sẽ tốt hơn.

## Ví dụ 4: Model tầng trên chỉ học từ dự đoán out-of-fold

Ta sinh 240 mẫu mới bằng một seed khác, và giữ 60 mẫu làm validation, đứng ngoài toàn bộ quá trình stacking. Với 180 mẫu train, ta chia ba fold. Ở mỗi fold, 60 mẫu được giữ lại, còn model gốc chỉ học trên 120 mẫu còn lại, rồi dự đoán cho 60 mẫu giữ lại. Nhờ vậy, mỗi mẫu train nhận đúng một dự đoán từ một model chưa từng thấy nó. Các dự đoán này, gọi là out-of-fold hay OOF, trở thành hai đặc trưng cho model tầng trên, là một logistic regression.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import log_loss

X, y = make_moons(n_samples=240, noise=0.30, random_state=184)
train_rows, val_rows = train_test_split(np.arange(len(y)), test_size=0.25,
                                       random_state=5, stratify=y)
Xt, yt, Xv, yv = X[train_rows], y[train_rows], X[val_rows], y[val_rows]
bases = [make_pipeline(StandardScaler(), LogisticRegression(max_iter=300)),
         DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=6)]
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=7).split(Xt, yt))
oof = np.zeros((len(yt), len(bases)))
filled = np.zeros_like(oof, dtype=int)
for fit_rows, held_rows in folds:
    assert not (set(fit_rows) & set(held_rows))
    for column, prototype in enumerate(bases):
        fitted = clone(prototype).fit(Xt[fit_rows], yt[fit_rows])
        assert fitted.classes_.tolist() == [0, 1]
        oof[held_rows, column] = fitted.predict_proba(Xt[held_rows])[:, 1]
        filled[held_rows, column] += 1
assert np.all(filled == 1)
meta = LogisticRegression(C=1.0, max_iter=300).fit(oof, yt)

full_models = [clone(prototype).fit(Xt, yt) for prototype in bases]
val_features = np.column_stack([model.predict_proba(Xv)[:, 1] for model in full_models])
assert not (set(train_rows) & set(val_rows))
stacked = meta.predict_proba(val_features)[:, 1]
print("OOF/validation feature shapes:", oof.shape, val_features.shape)
print("mỗi ô OOF được điền đúng một lần:", bool(np.all(filled == 1)))
for name, probability in [("linear", val_features[:, 0]), ("tree", val_features[:, 1]),
                          ("mean", val_features.mean(axis=1)), ("stack", stacked)]:
    distribution = np.column_stack([1 - probability, probability])
    print(name, "cùng validation log-loss:", f"{log_loss(yv, distribution):.4f}")

OOF/validation feature shapes: (180, 2) (60, 2)
mỗi ô OOF được điền đúng một lần: True
linear cùng validation log-loss: 0.3292
tree cùng validation log-loss: 2.6348
mean cùng validation log-loss: 0.3380
stack cùng validation log-loss: 0.3578


Bảng OOF có 180 hàng, hai cột, và mỗi ô được điền đúng một lần. Model tầng trên học từ bảng này. Khi dự đoán validation, model gốc được học lại trên toàn bộ 180 mẫu train, và dự đoán của chúng làm đặc trưng cho model tầng trên. Validation không tham gia vào bước học nào, ở cả hai tầng.

Lần chạy này, logistic đơn lẻ tốt nhất với 0.3292. Trung bình đạt 0.3380, stacking đạt 0.3578. Cây quyết định có log-loss tới 2.63, vì nó đưa ra những xác suất 0 hoặc 1 tuyệt đối mà lại sai. Một quy trình đúng vẫn có thể không cải thiện gì, và đó là kết quả bình thường, phải báo đúng như vậy.

Nếu dùng dự đoán của model gốc trên chính dữ liệu nó đã học để làm đặc trưng, model tầng trên sẽ thấy những dự đoán đẹp hơn thực tế, và học sai cách tin model gốc. Nếu dữ liệu có nhóm hay có thứ tự thời gian, phải chia fold OOF và chia validation theo đúng nhóm hay thời gian đó. Chia đều theo nhãn không tự giải quyết được rò rỉ kiểu này.